In [1]:
1

1

may 19

Cell 1: System Setup & Imports
This cell links the notebook to your isolated modules and loads the libraries.

In [5]:
import os
import sys
import numpy as np
import pandas as pd
import time

print("--- STEP 1: SYSTEM SETUP ---")
# Get the directory where the notebook is located (validation_code)
CURRENT_DIR = os.getcwd()
# Go one level up to find the 'api' folder
API_DIR = os.path.dirname(CURRENT_DIR) 

SERVICES_DIR = os.path.join(API_DIR, "services")

# THE FIX: Add the 'api' folder ITSELF to the Python path
if API_DIR not in sys.path: 
    sys.path.append(API_DIR)

# Add services folder so it can find the .dta files later
if SERVICES_DIR not in sys.path: 
    sys.path.append(SERVICES_DIR)

try:
    from ecd_nested_simulation_functions import generate_ecd_dummy_data
    from ecd_nested_simulation_functions import ecd_sampling_strategy
    print("✅ System paths registered.")
    print("✅ ECD Simulation modules imported successfully.\n")
except ImportError as e:
    print(f"❌ IMPORT ERROR: {e}")

--- STEP 1: SYSTEM SETUP ---
✅ System paths registered.
✅ ECD Simulation modules imported successfully.



Cell 2: Load WHO Biological Standards
This verifies that the .dta files are being read correctly from your services folder.

In [4]:
print("--- STEP 2: LOAD WHO STANDARDS ---")
def extract_data_from_dta(filename):
    file_path = os.path.join(SERVICES_DIR, filename)
    return pd.read_stata(file_path)

haz_params = extract_data_from_dta('lenanthro.dta')
waz_params = extract_data_from_dta('weianthro.dta')
whz_params_lying = extract_data_from_dta('wflanthro.dta')
whz_params_standing = extract_data_from_dta('wfhanthro.dta')

print("✅ WHO Standards Loaded Successfully.")
print(f"HAZ parameters shape: {haz_params.shape}\n")
display(haz_params.head(3)) # Visually verify the data

--- STEP 2: LOAD WHO STANDARDS ---
✅ WHO Standards Loaded Successfully.
HAZ parameters shape: (3714, 6)



,__000001,_agedays,l,m,s,loh
0,1,0,1,49.884201,0.03795,L
1,1,1,1,50.060101,0.03785,L
2,1,2,1,50.235901,0.03775,L


Cell 3: Generate the Synthetic Universe
This cell generates a scaled-down population so you can quickly test the math without waiting minutes.

In [8]:
print("--- STEP 3: GENERATE SYNTHETIC UNIVERSE ---")
N_L1S = 5       # Scaled down for rapid testing (normally 334)
N_L0S = 5       # Scaled down for rapid testing (normally 25)
N_CHILDREN = 5

common_real_params = {
    'girl_ratio': 0.5, 'min_age': 0, 'max_age': 1790, 'num_timepoints': 1, 'time_lags': [],
    'percent_stunting': 35, 'percent_underweight': 33, 'rho': 0.7
}

# Simulating a "Corrupt Collusion" Preset
L0_params_list, L1_params_list, L2_params_dict = generate_ecd_dummy_data.generate_nested_distortion_parameters(
    n_L1s=N_L1S, n_L0s_per_L1=N_L0S,
    real_percent_stunting=35, real_percent_underweight=33,
    mean_percent_under_reporting_stunting=30, mean_percent_under_reporting_underweight=30, 
    mean_bunch_factor_haz=0.25, mean_bunch_factor_waz=0.25, mean_bunch_factor_whz=0.25,
    mean_percent_copy=60, mean_collusion_index=0.90,
    sd_across_units_percent_under_reporting_stunting=2.0, sd_across_units_percent_under_reporting_underweight=2.0,
    sd_within_units_percent_under_reporting_stunting=1.0, sd_within_units_percent_under_reporting_underweight=1.0,
    sd_across_units_bunch_factor_haz=0.01, sd_across_units_bunch_factor_waz=0.01, sd_across_units_bunch_factor_whz=0.01,
    sd_within_units_bunch_factor_haz=0.01, sd_within_units_bunch_factor_waz=0.01, sd_within_units_bunch_factor_whz=0.01,
    sd_percent_copy=2.0, sd_collusion_index=0.02,
    error_sd_height_all_L0s=0.0, error_sd_weight_all_L0s=0.00,
    error_sd_height_L1=0.0, error_sd_weight_L1=0.00, 
    error_sd_height_L2=0.0, error_sd_weight_L2=0.00, 
    mean_time_lag_L1=15, mean_time_lag_L2=30
)

np.random.seed(42)
start = time.time()
nested_measurements = generate_ecd_dummy_data.generate_nested_measurements(
    real_params=common_real_params, L0_params_list=L0_params_list, L1_params_list=L1_params_list, L2_params_dict=L2_params_dict,
    n_L1s=N_L1S, n_L0s_per_L1=N_L0S, n_children_per_L0=N_CHILDREN, n_children_L1=N_CHILDREN, n_children_L2=N_CHILDREN,
    haz_params=haz_params, waz_params=waz_params, whz_params_lying=whz_params_lying, whz_params_standing=whz_params_standing, make_plots=False
)
print(f"✅ Synthetic Generation complete in {round(time.time()-start, 2)} seconds.\n")

--- STEP 3: GENERATE SYNTHETIC UNIVERSE ---


Assigning Personalities: 100%|██████████| 5/5 [00:00<?, ?it/s]


✅ Synthetic Generation complete in 3.39 seconds.



Cell 4: Data Flattening & Error Calculations
This flattens the nested dictionary into a readable Pandas DataFrame.

In [9]:
print("--- STEP 4: FLATTEN NESTED DATA ---")
all_children = []

for L1_id, L1_data in nested_measurements.items():
    if L1_id == 'metadata': continue
    for L0_id, L0_data in L1_data.items():
        if L0_id == 'L1_info': continue
        
        df_real = L0_data['real']['data'].copy().rename(columns={'haz': 'real_haz'})
        df_L0 = L0_data['L0']['data'].copy().rename(columns={'haz': 'L0_haz'})
        df_L1 = L0_data['L1']['data'].copy().rename(columns={'haz': 'L1_haz'})
        df_L2 = L0_data['L2']['data'].copy().rename(columns={'haz': 'L2_haz'})
        
        merged = df_real[['child_id', 'real_haz']].merge(
            df_L0[['child_id', 'L0_haz']], on='child_id', how='left').merge(
            df_L1[['child_id', 'L1_haz']], on='child_id', how='left').merge(
            df_L2[['child_id', 'L2_haz']], on='child_id', how='left')
            
        merged['L1_id'] = L1_id
        merged['L0_id'] = L0_id
        all_children.append(merged)

df_pop = pd.concat(all_children, ignore_index=True)

# Calculate individual errors
df_pop['L0_Error'] = np.abs(df_pop['L0_haz'] - df_pop['real_haz'])
df_pop['L1_Error'] = np.abs(df_pop['L1_haz'] - df_pop['L0_haz'])

print(f"✅ Flattening complete. Total children generated: {len(df_pop)}")
display(df_pop.head())

--- STEP 4: FLATTEN NESTED DATA ---
✅ Flattening complete. Total children generated: 125


,child_id,real_haz,L0_haz,L1_haz,L2_haz,L1_id,L0_id,L0_Error,L1_Error
0,L1_0_L0_0_child_0,-0.638898,-0.638898,-0.638898,-0.638898,L1_0,L0_0,9.992007e-16,1.665335e-15
1,L1_0_L0_0_child_1,-1.233308,-1.233308,-1.317069,-1.233308,L1_0,L0_0,1.332268e-15,8.376171e-02
2,L1_0_L0_0_child_2,-1.875282,-1.875282,-1.983216,-1.875282,L1_0,L0_0,4.440892e-16,1.079342e-01
3,L1_0_L0_0_child_3,-3.558867,-1.002671,-3.558867,-3.558867,L1_0,L0_0,2.556196e+00,2.556196e+00
4,L1_0_L0_0_child_4,0.400353,0.400353,0.278471,0.400353,L1_0,L0_0,1.332268e-15,1.218818e-01


Cell 5: Strategy Calculation Engine
This is where the magic happens: calculating the "God Mode" truth, running L1's limited sample, and seeing how accurate L1 is.

In [10]:
print("--- STEP 5: STRATEGY CALCULATION ENGINE ---")

def calculate_metrics(df, group_cols, col_meas, col_baseline):
    temp = df.copy()
    temp['_err'] = np.abs(temp[col_meas] - temp[col_baseline])
    return temp.groupby(group_cols).agg(MAE=('_err', 'mean')).reset_index()

TARGET_L1_COUNT = max(1, int(N_L1S * 0.30)) # Target worst 30%
print(f"Goal: Catch the top {TARGET_L1_COUNT} worst L1 Supervisors.")

# 1. GOD MODE: The absolute truth
god_l1 = calculate_metrics(df_pop, ['L1_id'], 'L0_haz', 'real_haz')
god_l1_mae = set(god_l1.sort_values(by='MAE', ascending=False).head(TARGET_L1_COUNT)['L1_id'])
print(f"God Mode Worst L1s (Top 3): {list(god_l1_mae)[:3]}...")

# 2. L1 Sampling
L1_BUDGET_PCT = 0.40 # Testing a 40% Budget
TOTAL_KIDS = N_L0S * N_CHILDREN
budget_k = int(TOTAL_KIDS * L1_BUDGET_PCT)

# Balanced strategy: e.g., 5 clinics x 12 kids = 60 kids (40% of 150)
l1_c, l1_k = min(5, N_L0S), min(12, N_CHILDREN)
print(f"\n--- Testing L1 Strategy: {l1_c} Clinics x {l1_k} Kids ({L1_BUDGET_PCT*100}% Budget) ---")

# Execute Bulletproof Sampling
l1_clinics = df_pop[['L1_id', 'L0_id']].drop_duplicates().groupby('L1_id').apply(
    lambda x: x.sample(n=min(len(x), l1_c), replace=False)).reset_index(drop=True)

df_l1_sheet = df_pop.merge(l1_clinics, on=['L1_id', 'L0_id']).groupby(['L1_id', 'L0_id']).apply(
    lambda x: x.sample(n=min(len(x), l1_k), replace=False)).reset_index(drop=True)

print(f"L1 successfully sampled {len(df_l1_sheet)} total kids across the region.")

l1_diagnosis = calculate_metrics(df_l1_sheet, ['L1_id'], 'L1_haz', 'L0_haz')
l1_caught = set(l1_diagnosis.sort_values(by='MAE', ascending=False).head(TARGET_L1_COUNT)['L1_id'])
v1_acc = (len(l1_caught & god_l1_mae) / TARGET_L1_COUNT) * 100

print(f"👉 L1 Ranking Accuracy: {round(v1_acc, 1)}%")

--- STEP 5: STRATEGY CALCULATION ENGINE ---
Goal: Catch the top 1 worst L1 Supervisors.
God Mode Worst L1s (Top 3): ['L1_0']...

--- Testing L1 Strategy: 5 Clinics x 5 Kids (40.0% Budget) ---
L1 successfully sampled 125 total kids across the region.
👉 L1 Ranking Accuracy: 100.0%


Cell 6: L2 Auditing Validation
This proves L2's ability to catch corrupt L1s using a limited sample.

In [11]:
# 3. L2 Auditing
L2_BUDGET_PCT = 0.50 # L2 audits 50% of L1's sheet
l2_c, l2_k = min(3, l1_c), min(10, l1_k)
print(f"\n--- Testing L2 Execution: {l2_c} Clinics x {l2_k} Kids ---")

df_l2_audit = df_l1_sheet.groupby('L1_id').apply(
    lambda x: x.sample(n=min(len(x), l2_c * l2_k), replace=False)).reset_index(drop=True)

sheet_truth = calculate_metrics(df_l1_sheet, ['L1_id'], 'L1_haz', 'real_haz')
sheet_mae = set(sheet_truth.sort_values(by='MAE', ascending=False).head(TARGET_L1_COUNT)['L1_id'])

l2_diagnosis = calculate_metrics(df_l2_audit, ['L1_id'], 'L2_haz', 'L1_haz')
l2_caught = set(l2_diagnosis.sort_values(by='MAE', ascending=False).head(TARGET_L1_COUNT)['L1_id'])
v3_acc = (len(l2_caught & sheet_mae) / TARGET_L1_COUNT) * 100

print(f"👉 L2 Ranking Accuracy (Catching corrupt L1s): {round(v3_acc, 1)}%")
print("\n✅ VALIDATION COMPLETE.")


--- Testing L2 Execution: 3 Clinics x 5 Kids ---
👉 L2 Ranking Accuracy (Catching corrupt L1s): 100.0%

✅ VALIDATION COMPLETE.
